### Data Exploration

Exploration of PhysionetMI dataset to determine preprocessing parameters.
Each section investigates one pipeline decision. Final decisions in decision log 17 May 2026.

Sections: PSD and frequency range, sampling rate check, re-referencing, 
artifact rejection, channel selection, normalisation.

In [ ]:
import mne
from mne.preprocessing import ICA, create_eog_epochs
from mne_icalabel import label_components

import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
import re

from sklearn.preprocessing import RobustScaler

import torch

# Reproducibility
np.random.seed(25)
random.seed(25)
torch.manual_seed(25)

mne.set_log_level("WARNING")

In [ ]:
data_path = Path("../Data/raw/physionet_mi_data/eeg-motor-movementimagery-dataset")

In [ ]:
subject = "S001"
run = "R04"

edf_path= data_path / subject / f"{subject}{run}.edf"
raw = mne.io.read_raw_edf(edf_path, preload=True)

print("Sampling rate:", raw.info['sfreq'], "Hz")
print("Number of channels:", len(raw.ch_names))
print("Channel names:", raw.ch_names)
print("Duration:", round(raw.times[-1],2), "seconds")

In [ ]:
events, event_id = mne.events_from_annotations(raw)

print("Event IDs:", event_id)
print("Total events:", len(events))
print("First 10 events:", events[:10])

### Extract epochs (trials) from the raw data

In [ ]:

epochs = mne.Epochs(
    raw,
    events,
    event_id={'T1': 2, 'T2': 3},
    tmin=0,
    tmax=3,
    baseline=None,
    preload=True
)

print(epochs)

In [ ]:
# Plot the first trial, first channel
trial = epochs['T1'][0]
data = trial.get_data()[0, 0, :]
times = trial.times

plt.figure(figsize=(10, 4))
plt.plot(times, data * 1e6)
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude (microvolts)')
plt.title('Single Motor Imagery Trial - Subject 1, Left Hand, Channel 1')
plt.grid(True)
plt.show()

In [ ]:
print(f"Min amplitude: {data.min() * 1e6:.2f} microvolts")
print(f"Max amplitude: {data.max() * 1e6:.2f} microvolts")
print(f"Mean amplitude: {data.mean() * 1e6:.2f} microvolts")
print(f"Data shape: {data.shape}")

### Load same run from subject 2

In [ ]:
subject = "S002"
run = "R04"

edf_path2= data_path / subject / f"{subject}{run}.edf"
raw2 = mne.io.read_raw_edf(edf_path2, preload=True)

events2, event_id2 = mne.events_from_annotations(raw2)

epochs2 = mne.Epochs(
    raw2,
    events2,
    event_id={'T1': 2, 'T2': 3},
    tmin=0,
    tmax=3,
    baseline=None,
    preload=True
)

# Get first left hand trial from each subject
trial1 = epochs['T1'][0].get_data()[0, 0, :]
trial2 = epochs2['T1'][0].get_data()[0, 0, :]
times = epochs['T1'][0].times

# Plot both on same figure
plt.figure(figsize=(12, 5))
plt.plot(times, trial1 * 1e6, label='Subject 1', alpha=0.7)
plt.plot(times, trial2 * 1e6, label='Subject 2', alpha=0.7)
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude (microvolts)')
plt.title('Same Motor Imagery Task - Subject 1 vs Subject 2, Channel 1')
plt.legend()
plt.grid(True)
plt.show()

### PSD exploration - checking frequency content

In [ ]:
fig = raw.compute_psd(fmax=80).plot(show=False)
plt.title("PSD - S001 R04 - Unfiltered - All channels")
plt.tight_layout()
plt.show()

# Print numerical summary of power in key bands
psd = raw.compute_psd(fmax=80)
freqs = psd.freqs
power = psd.get_data().mean(axis=0)  # average across all channels

# Define bands
bands = {
    'delta (1-4 Hz)': (1, 4),
    'theta (4-8 Hz)': (4, 8),
    'mu/alpha (8-13 Hz)': (8, 13),
    'beta (13-30 Hz)': (13, 30),
    'gamma (30-80 Hz)': (30, 80)
}

print("Average power by frequency band (unfiltered, all channels):")
for band_name, (fmin, fmax_band) in bands.items():
    idx = (freqs >= fmin) & (freqs <= fmax_band)
    mean_power = power[idx].mean()
    # sanity check, remove in final pipeline
    print(f"  {band_name}: {mean_power:.2f}")

### outlier channel visible in PSD

In [ ]:
# sanity check, remove in final pipeline

psd = raw.compute_psd(fmax=80)# Identify outlier channels visible in PSD
# sanity check, remove in final pipeline

psd = raw.compute_psd(fmax=80)
freqs = psd.freqs
power_per_channel = psd.get_data()  # shape: channels x freqs

# Check motor imagery band 15-30 Hz
idx_15_30 = (freqs >= 15) & (freqs <= 30)
mean_power_15_30 = power_per_channel[:, idx_15_30].mean(axis=1)

print("Top 3 channels by mean power in 15-30 Hz:")
top_idx = np.argsort(mean_power_15_30)[::-1][:3]
for i, ch_idx in enumerate(top_idx):
    print(f"  {i+1}. {raw.ch_names[ch_idx]}: {mean_power_15_30[ch_idx]:.2f}")

# Check high frequency range 50-80 Hz where outliers appeared
idx_50_80 = (freqs >= 50) & (freqs <= 80)
mean_power_50_80 = power_per_channel[:, idx_50_80].mean(axis=1)

print("\nTop 3 channels by mean power in 50-80 Hz:")
top_idx_high = np.argsort(mean_power_50_80)[::-1][:3]
for i, ch_idx in enumerate(top_idx_high):
    print(f"  {i+1}. {raw.ch_names[ch_idx]}: {mean_power_50_80[ch_idx]:.2f}")

# Check if any motor cortex channels appear in either list
motor_channels = ['C3..', 'Cz..', 'C4..']
print("\nMotor cortex channel power in 15-30 Hz:")
for ch in motor_channels:
    ch_idx = raw.ch_names.index(ch)
    print(f"  {ch}: {mean_power_15_30[ch_idx]:.2f}")

print("\nMotor cortex channel power in 50-80 Hz:")
for ch in motor_channels:
    ch_idx = raw.ch_names.index(ch)
    print(f"  {ch}: {mean_power_50_80[ch_idx]:.2f}")
freqs = psd.freqs
power_per_channel = psd.get_data()  # shape: channels x freqs

# Find channel with highest mean power in 15-30 Hz range
idx_15_30 = (freqs >= 15) & (freqs <= 30)
mean_power_15_30 = power_per_channel[:, idx_15_30].mean(axis=1)

# Get top 3 channels by power in this range
top_channels_idx = np.argsort(mean_power_15_30)[::-1][:3]
print("Top 3 channels by mean power in 15-30 Hz:")
for i, ch_idx in enumerate(top_channels_idx):
    print(f"  {i+1}. {raw.ch_names[ch_idx]}: {mean_power_15_30[ch_idx]:.2f}")

# Also print the lowest 3 for comparison
bottom_channels_idx = np.argsort(mean_power_15_30)[:3]
print("\nBottom 3 channels by mean power in 15-30 Hz:")
for i, ch_idx in enumerate(bottom_channels_idx):
    print(f"  {i+1}. {raw.ch_names[ch_idx]}: {mean_power_15_30[ch_idx]:.2f}")

### any subjects with incompatible or unexpected sampling rates

In [ ]:
# sampling rate check across all subjects

from pathlib import Path

sampling_rates = {}
subjects_with_issues = []

for s_num in range(1, 110):
    subject = f"S{s_num:03d}"
    edf_path = data_path / subject / f"{subject}R04.edf"

    if not edf_path.exists():
        print(f"{subject}: file not found")
        subjects_with_issues.append(subject)
        continue

    raw_check = mne.io.read_raw_edf(edf_path, preload=False)
    sfreq = raw_check.info['sfreq']
    sampling_rates[subject] = sfreq

# summarise results
from collections import Counter
rate_counts = Counter(sampling_rates.values())

print("Sampling rate distribution across all subjects:")
for rate, count in sorted(rate_counts.items()):
    print(f"  {rate} Hz: {count} subjects")

print("\nSubjects with sampling rate different from 160 Hz:")
for subject, rate in sampling_rates.items():
    if rate != 160.0:
        print(f"  {subject}: {rate} Hz")

if subjects_with_issues:
    print("\nSubjects with missing files:")
    for subject in subjects_with_issues:
        print(f"  {subject}")

### final subject list

In [ ]:
# final subject list excluding incompatible sampling rates: S088, S092, S100 excluded due to 128 Hz sampling rate

all_subjects = [f"S{i:03d}" for i in range(1, 110)]
excluded = ['S088', 'S092', 'S100']
valid_subjects = [s for s in all_subjects if s not in excluded]

print(f"Total subjects in dataset: {len(all_subjects)}")
print(f"Excluded subjects: {excluded}")
print(f"Valid subjects for analysis: {len(valid_subjects)}")

### PSD check across 10 subjects

In [ ]:
# checking consistency of frequency content before finalising filter parameters

selected_subjects = valid_subjects[::10][:10]
print("Subjects selected for PSD check:", selected_subjects)

run = 'R04'
band_summary = {}

fig, axes = plt.subplots(len(selected_subjects), 1, figsize=(12, 25))

bands = {
    'delta (1-4 Hz)': (1, 4),
    'theta (4-8 Hz)': (4, 8),
    'mu (8-13 Hz)': (8, 13),
    'beta (13-30 Hz)': (13, 30),
    'gamma (30-80 Hz)': (30, 80)
}

for i, subject in enumerate(selected_subjects):
    edf_path = data_path / subject / f"{subject}{run}.edf"
    raw_sub = mne.io.read_raw_edf(edf_path, preload=True)

    sfreq = raw_sub.info['sfreq']
    nyquist = sfreq / 2
    fmax_sub = min(80, nyquist - 1)

    psd = raw_sub.compute_psd(fmax=fmax_sub)
    freqs = psd.freqs
    power = psd.get_data().mean(axis=0)

    axes[i].plot(freqs, 10 * np.log10(power))
    axes[i].set_title(f"{subject} - sfreq: {sfreq} Hz")
    axes[i].set_xlabel('Frequency (Hz)')
    axes[i].set_ylabel('Power (dB/Hz)')
    axes[i].axvline(x=60, color='r', linestyle='--', alpha=0.5)
    axes[i].axvspan(8, 30, alpha=0.1, color='blue')
    axes[i].grid(True)

    band_summary[subject] = {}
    for band_name, (fmin, fmax_band) in bands.items():
        idx = (freqs >= fmin) & (freqs <= fmax_band)
        band_summary[subject][band_name] = power[idx].mean()

plt.tight_layout()
plt.show()

### band power summary

In [ ]:
print(f"\n{'Band':<20}", end="")
for subject in selected_subjects:
    print(f"{subject:>10}", end="")
print()

for band_name in bands.keys():
    print(f"{band_name:<20}", end="")
    for subject in selected_subjects:
        print(f"{band_summary[subject][band_name]:>10.2f}", end="")
    print()

### Re-referencing exploration

In [ ]:
# comparing common average, linked mastoid, and laplacian on S001 R04
# laplacian uses standard 10-10 montage as digitization not available in dataset

motor_channels = ['C3..', 'Cz..', 'C4..']

# load fresh unfiltered raw for fair comparison
raw_ref = mne.io.read_raw_edf(
    data_path / 'S001' / 'S001R04.edf',
    preload=True
)

# apply notch and bandpass first as these come before re-referencing in pipeline
raw_ref.notch_filter(freqs=60)
raw_ref.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')

# scheme 1: common average reference
raw_car = raw_ref.copy().set_eeg_reference('average', projection=False)

# scheme 2: linked mastoid reference
raw_lm = raw_ref.copy().set_eeg_reference(['T9..', 'T10.'], projection=False)

In [ ]:
# initial montage assignment check
# this shows the baseline mismatch before the fix is applied

raw_check = raw_ref.copy()
montage = mne.channels.make_standard_montage('standard_1020')

# strip dots only, no capitalisation fix
rename_map_check = {ch: ch.rstrip('.') for ch in raw_check.ch_names}
raw_check.rename_channels(rename_map_check)
raw_check.set_montage(montage, on_missing='ignore')

no_position = []
has_position = []

for ch in raw_check.info['chs']:
    pos = ch['loc'][:3]
    if not np.isfinite(pos).all() or np.isclose(pos, 0.0).all():
        no_position.append(ch['ch_name'])
    else:
        has_position.append(ch['ch_name'])

print(f"Channels with positions assigned: {len(has_position)}")
print(f"Channels without positions: {len(no_position)}")
print("\nChannels missing positions:")

print(no_position)
# for ch in no_position:
#     print(f"  {ch}")

#### Laplacian

In [ ]:
# scheme 3: laplacian with standard 10-10 montage
raw_lap = raw_ref.copy()
montage = mne.channels.make_standard_montage('standard_1020')

# strip dots first
rename_map = {ch: ch.rstrip('.') for ch in raw_lap.ch_names}
raw_lap.rename_channels(rename_map)

# build case-insensitive lookup from actual montage channel names
# avoids guessing capitalisation rules
montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

def fix_channel_name(name):
    upper = name.upper()
    if upper in montage_lookup:
        return montage_lookup[upper]
    return name

capitalisation_fix = {ch: fix_channel_name(ch) for ch in raw_lap.ch_names}

# sanity check, remove in final pipeline
print("Sample name fixes:")
for old, new in list(capitalisation_fix.items())[:10]:
    print(f"  {old} -> {new}")

raw_lap.rename_channels(capitalisation_fix)
raw_lap.set_montage(montage, on_missing='ignore')

# check positions
no_position = []
has_position = []
for ch in raw_lap.info['chs']:
    pos = ch['loc'][:3]
    if not np.isfinite(pos).all() or np.isclose(pos, 0.0).all():
        no_position.append(ch['ch_name'])
    else:
        has_position.append(ch['ch_name'])

print(f"\nChannels with positions: {len(has_position)}")
print(f"Channels without positions: {len(no_position)}")
if no_position:
    print("Missing positions:", no_position)

motor_check = ['C3', 'Cz', 'C4']
motor_ok = all(ch in has_position for ch in motor_check)
print(f"\nMotor cortex channels have positions: {motor_ok}")

In [ ]:
# compute laplacian, drop channels without positions first
mne.preprocessing.compute_current_source_density(raw_lap)

# rename back to dotted names for consistency with rest of pipeline
rename_back = {v: k for k, v in rename_map.items()}

# only rename channels that exist in current ch_names
rename_back_valid = {k: v for k, v in rename_back.items()
                     if k in raw_lap.ch_names}
raw_lap.rename_channels(rename_back_valid)

print("Laplacian computed successfully")
print("Motor cortex channels present:",
      all(ch in raw_lap.ch_names for ch in motor_channels))

### PSD comparison of motor cortex channels under three referencing schemes

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

schemes = {
    'common average': raw_car,
    'linked mastoid': raw_lm,
    'laplacian': raw_lap
}

psd_summary = {}

for ax, (scheme_name, raw_scheme) in zip(axes, schemes.items()):
    psd = raw_scheme.compute_psd(fmax=40, picks=motor_channels)
    freqs = psd.freqs
    power = psd.get_data()

    for ch_idx, ch_name in enumerate(motor_channels):
        ax.plot(freqs, 10 * np.log10(power[ch_idx]), label=ch_name)

    ax.set_title(scheme_name)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power (dB/Hz)')
    ax.legend()
    ax.grid(True)

    psd_summary[scheme_name] = {}
    idx_mu = (freqs >= 8) & (freqs <= 13)
    idx_beta = (freqs >= 13) & (freqs <= 30)

    psd_summary[scheme_name]['mu'] = power[:, idx_mu].mean(axis=1)
    psd_summary[scheme_name]['beta'] = power[:, idx_beta].mean(axis=1)

plt.tight_layout()
plt.show()

In [ ]:
# numerical summary of motor cortex channel power under each scheme

print(f"{'Channel':<10} {'Scheme':<20} {'Mu mean power':>15} {'Beta mean power':>15}")
print("-" * 62)
for scheme_name in schemes.keys():
    for ch_idx, ch_name in enumerate(motor_channels):
        print(
            f"{ch_name:<10} "
            f"{scheme_name:<20} "
            f"{psd_summary[scheme_name]['mu'][ch_idx]:>15.2f} "
            f"{psd_summary[scheme_name]['beta'][ch_idx]:>15.2f}"
        )
    print()

### C3 vs C4 separation during motor imagery trials

In [ ]:
# C3 vs C4 separation during motor imagery trials under each referencing scheme
# checking whether left vs right hand imagery produces distinct lateral patterns
# T1 = left fist (should activate C4), T2 = right fist (should activate C3)

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

separation_summary = {}

for row, (scheme_name, raw_scheme) in enumerate(schemes.items()):

    # extract epochs for this scheme
    events_scheme, _ = mne.events_from_annotations(raw_scheme)
    epochs_scheme = mne.Epochs(
        raw_scheme,
        events_scheme,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    # get mean signal for C3 and C4 across trials
    # picks only C3 and C4, not Cz
    c3_idx = epochs_scheme.ch_names.index('C3..')
    c4_idx = epochs_scheme.ch_names.index('C4..')

    # mean across time for each trial
    t1_c3 = epochs_scheme['T1'].get_data()[:, c3_idx, :].mean(axis=1)
    t1_c4 = epochs_scheme['T1'].get_data()[:, c4_idx, :].mean(axis=1)
    t2_c3 = epochs_scheme['T2'].get_data()[:, c3_idx, :].mean(axis=1)
    t2_c4 = epochs_scheme['T2'].get_data()[:, c4_idx, :].mean(axis=1)

    # mean time series for each condition
    t1_c3_ts = epochs_scheme['T1'].get_data()[:, c3_idx, :].mean(axis=0)
    t1_c4_ts = epochs_scheme['T1'].get_data()[:, c4_idx, :].mean(axis=0)
    t2_c3_ts = epochs_scheme['T2'].get_data()[:, c3_idx, :].mean(axis=0)
    t2_c4_ts = epochs_scheme['T2'].get_data()[:, c4_idx, :].mean(axis=0)
    times = epochs_scheme.times

    # plot left column: T1 trials (left fist)
    axes[row, 0].plot(times, t1_c3_ts * 1e6, label='C3', color='blue')
    axes[row, 0].plot(times, t1_c4_ts * 1e6, label='C4', color='red')
    axes[row, 0].set_title(f'{scheme_name} - T1 left fist')
    axes[row, 0].set_xlabel('Time (s)')
    axes[row, 0].set_ylabel('Amplitude (uV)')
    axes[row, 0].legend()
    axes[row, 0].grid(True)

    # plot right column: T2 trials (right fist)
    axes[row, 1].plot(times, t2_c3_ts * 1e6, label='C3', color='blue')
    axes[row, 1].plot(times, t2_c4_ts * 1e6, label='C4', color='red')
    axes[row, 1].set_title(f'{scheme_name} - T2 right fist')
    axes[row, 1].set_xlabel('Time (s)')
    axes[row, 1].set_ylabel('Amplitude (uV)')
    axes[row, 1].legend()
    axes[row, 1].grid(True)

    # numerical separation: difference between C3 and C4 mean amplitude
    separation_summary[scheme_name] = {
        'T1 C3-C4 diff': abs(t1_c3.mean() - t1_c4.mean()),
        'T2 C3-C4 diff': abs(t2_c3.mean() - t2_c4.mean()),
        'T1 trials': len(epochs_scheme['T1']),
        'T2 trials': len(epochs_scheme['T2'])
    }

plt.tight_layout()
plt.show()

In [ ]:
# numerical summary
print(f"{'Scheme':<20} {'T1 C3-C4 diff':>15} {'T2 C3-C4 diff':>15} {'T1 trials':>10} {'T2 trials':>10}")
print("-" * 72)
for scheme_name, vals in separation_summary.items():
    print(
        f"{scheme_name:<20} "
        f"{vals['T1 C3-C4 diff']:>15.2f} "
        f"{vals['T2 C3-C4 diff']:>15.2f} "
        f"{vals['T1 trials']:>10} "
        f"{vals['T2 trials']:>10}"
    )

### C3 vs C4 separation metric using variance

In [ ]:
# mean amplitude cancels out for oscillatory signals, variance does not

print(f"{'Scheme':<20} {'T1 C3 var':>12} {'T1 C4 var':>12} {'T2 C3 var':>12} {'T2 C4 var':>12} {'T1 ratio':>10} {'T2 ratio':>10}")
print("-" * 90)

variance_summary = {}

for scheme_name, raw_scheme in schemes.items():
    events_scheme, _ = mne.events_from_annotations(raw_scheme)
    epochs_scheme = mne.Epochs(
        raw_scheme,
        events_scheme,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    c3_idx = epochs_scheme.ch_names.index('C3..')
    c4_idx = epochs_scheme.ch_names.index('C4..')

    # variance across time per trial, then mean across trials
    t1_c3_var = epochs_scheme['T1'].get_data()[:, c3_idx, :].var(axis=1).mean()
    t1_c4_var = epochs_scheme['T1'].get_data()[:, c4_idx, :].var(axis=1).mean()
    t2_c3_var = epochs_scheme['T2'].get_data()[:, c3_idx, :].var(axis=1).mean()
    t2_c4_var = epochs_scheme['T2'].get_data()[:, c4_idx, :].var(axis=1).mean()

    # ratio of C3 to C4 variance
    # for T1 left fist: expect C4 higher (contralateral activation)
    # for T2 right fist: expect C3 higher
    t1_ratio = t1_c3_var / t1_c4_var
    t2_ratio = t2_c3_var / t2_c4_var

    variance_summary[scheme_name] = {
        't1_c3': t1_c3_var,
        't1_c4': t1_c4_var,
        't2_c3': t2_c3_var,
        't2_c4': t2_c4_var,
        't1_ratio': t1_ratio,
        't2_ratio': t2_ratio
    }

    print(
        f"{scheme_name:<20} "
        f"{t1_c3_var:>12.2f} "
        f"{t1_c4_var:>12.2f} "
        f"{t2_c3_var:>12.2f} "
        f"{t2_c4_var:>12.2f} "
        f"{t1_ratio:>10.3f} "
        f"{t2_ratio:>10.3f}"
    )

### Artifact Rejection

In [ ]:
# step 1: look at amplitude distribution across trials to understand
# what clean vs corrupted trials look like in this dataset

# load S001 for exploration
raw_art = mne.io.read_raw_edf(
    data_path / 'S001' / 'S001R04.edf',
    preload=True
)

# pipeline steps that come before artifact rejection
raw_art.notch_filter(freqs=60)
raw_art.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
raw_art.set_eeg_reference('average', projection=False)

# extract epochs
events_art, _ = mne.events_from_annotations(raw_art)
epochs_art = mne.Epochs(
    raw_art,
    events_art,
    event_id={'T1': 2, 'T2': 3},
    tmin=0,
    tmax=3,
    baseline=None,
    preload=True
)

# get peak to peak amplitude per trial per channel
# peak to peak = max value minus min value within each trial
data = epochs_art.get_data()  # shape: trials x channels x timepoints
peak_to_peak = data.max(axis=2) - data.min(axis=2)  # trials x channels

# max peak to peak across all channels per trial
# this gives one number per trial representing its worst channel
max_ptp_per_trial = peak_to_peak.max(axis=1)

print(f"Number of trials: {len(max_ptp_per_trial)}")
print(f"Peak to peak amplitude stats across trials (microvolts):")
print(f"  min:    {max_ptp_per_trial.min() * 1e6:.1f}")
print(f"  max:    {max_ptp_per_trial.max() * 1e6:.1f}")
print(f"  mean:   {max_ptp_per_trial.mean() * 1e6:.1f}")
print(f"  median: {np.median(max_ptp_per_trial) * 1e6:.1f}")
print(f"  std:    {max_ptp_per_trial.std() * 1e6:.1f}")

# show per trial values so we can see the distribution
print(f"\nPeak to peak per trial (microvolts):")
for i, val in enumerate(max_ptp_per_trial):
    print(f"  Trial {i+1:2d}: {val * 1e6:.1f} uV")

In [ ]:
# plot amplitude distribution to visualise the spread
# and check where natural cutoff might sit

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left plot: per trial amplitudes as bar chart
axes[0].bar(range(1, len(max_ptp_per_trial) + 1),
            max_ptp_per_trial * 1e6)
axes[0].axhline(y=200, color='r', linestyle='--', alpha=0.7)
axes[0].axhline(y=250, color='orange', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Trial number')
axes[0].set_ylabel('Peak to peak amplitude (uV)')
axes[0].set_title('S001 R04 - peak to peak per trial')
axes[0].legend(['200 uV threshold', '250 uV threshold'])
axes[0].grid(True)

# right plot: histogram of amplitudes
axes[1].hist(max_ptp_per_trial * 1e6, bins=10, edgecolor='black')
axes[1].axvline(x=200, color='r', linestyle='--', alpha=0.7)
axes[1].axvline(x=250, color='orange', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Peak to peak amplitude (uV)')
axes[1].set_ylabel('Number of trials')
axes[1].set_title('S001 R04 - amplitude distribution')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# show how many trials would be rejected at each threshold
thresholds = [150, 175, 200, 225, 250, 275]
print("Trials rejected at different thresholds:")
print(f"{'Threshold (uV)':<20} {'Rejected':>10} {'Remaining':>10} {'Percent kept':>15}")
print("-" * 57)
for thresh in thresholds:
    rejected = (max_ptp_per_trial * 1e6 > thresh).sum()
    remaining = len(max_ptp_per_trial) - rejected
    pct = remaining / len(max_ptp_per_trial) * 100
    print(f"{thresh:<20} {rejected:>10} {remaining:>10} {pct:>14.1f}%")

In [ ]:
# check amplitude distribution across multiple subjects and all three runs
# to see if 200 uV threshold is consistent or needs adjustment

subjects_to_check = ['S001', 'S010', 'S020', 'S030', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']

print(f"{'Subject':<10} {'Run':<6} {'Trials':<8} {'Min uV':<10} {'Max uV':<10} {'Median uV':<12} {'Above 200uV':<12} {'Above 250uV':<12}")
print("-" * 82)

all_ptp = []

for subject in subjects_to_check:
    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"

        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)

        events_s, _ = mne.events_from_annotations(raw_s)
        epochs_s = mne.Epochs(
            raw_s,
            events_s,
            event_id={'T1': 2, 'T2': 3},
            tmin=0,
            tmax=3,
            baseline=None,
            preload=True
        )

        data_s = epochs_s.get_data()
        ptp_s = (data_s.max(axis=2) - data_s.min(axis=2)).max(axis=1)
        all_ptp.extend(ptp_s * 1e6)

        above_200 = (ptp_s * 1e6 > 200).sum()
        above_250 = (ptp_s * 1e6 > 250).sum()

        print(
            f"{subject:<10} {run:<6} {len(ptp_s):<8} "
            f"{ptp_s.min() * 1e6:<10.1f} "
            f"{ptp_s.max() * 1e6:<10.1f} "
            f"{np.median(ptp_s) * 1e6:<12.1f} "
            f"{above_200:<12} "
            f"{above_250:<12}"
        )

all_ptp = np.array(all_ptp)
print(f"\nOverall across all subjects and runs:")
print(f"  Total trials: {len(all_ptp)}")
print(f"  Min: {all_ptp.min():.1f} uV")
print(f"  Max: {all_ptp.max():.1f} uV")
print(f"  Median: {np.median(all_ptp):.1f} uV")
print(f"  Above 200 uV: {(all_ptp > 200).sum()} trials ({(all_ptp > 200).mean() * 100:.1f}%)")
print(f"  Above 250 uV: {(all_ptp > 250).sum()} trials ({(all_ptp > 250).mean() * 100:.1f}%)")

In [ ]:
# S020 in more detail
# amplitude seems unusually low compared to other subjects

raw_s20 = mne.io.read_raw_edf(
    data_path / 'S020' / 'S020R04.edf',
    preload=True
)
raw_s20.notch_filter(freqs=60)
raw_s20.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
raw_s20.set_eeg_reference('average', projection=False)

# check PSD for S020 vs S001
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (subject, raw_plot) in zip(axes, [('S001', raw_art), ('S020', raw_s20)]):
    psd = raw_plot.compute_psd(fmax=40, picks=motor_channels)
    freqs = psd.freqs
    power = psd.get_data()
    for ch_idx, ch_name in enumerate(motor_channels):
        ax.plot(freqs, 10 * np.log10(power[ch_idx]), label=ch_name)
    ax.set_title(f'{subject} - motor cortex PSD after filtering')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power (dB/Hz)')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

# print amplitude stats for S020 motor cortex channels
events_s20, _ = mne.events_from_annotations(raw_s20)
epochs_s20 = mne.Epochs(
    raw_s20,
    events_s20,
    event_id={'T1': 2, 'T2': 3},
    tmin=0,
    tmax=3,
    baseline=None,
    preload=True
)

data_s20 = epochs_s20.get_data()
print("S020 motor cortex channel amplitude stats (microvolts):")
for ch_name in motor_channels:
    ch_idx = epochs_s20.ch_names.index(ch_name)
    ch_data = data_s20[:, ch_idx, :] * 1e6
    print(f"  {ch_name}: min {ch_data.min():.1f}, max {ch_data.max():.1f}, std {ch_data.std():.1f}")

print("\nS001 motor cortex channel amplitude stats (microvolts):")
data_s01 = epochs_art.get_data()
for ch_name in motor_channels:
    ch_idx = epochs_art.ch_names.index(ch_name)
    ch_data = data_s01[:, ch_idx, :] * 1e6
    print(f"  {ch_name}: min {ch_data.min():.1f}, max {ch_data.max():.1f}, std {ch_data.std():.1f}")

In [ ]:
# subject relative threshold exploration
# testing multipliers of 2x, 3x, 4x the subject median

subjects_to_check = ['S001', 'S010', 'S020', 'S030', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']
multipliers = [2, 3, 4]

print(f"{'Subject':<10} {'Median uV':<12}", end="")
for m in multipliers:
    print(f"{'Thresh ' + str(m) + 'x':<12} {'Rejected':<10} {'Pct kept':<10}", end="")
print()
print("-" * 100)

subject_results = {}

for subject in subjects_to_check:
    all_ptp_subject = []

    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)

        events_s, _ = mne.events_from_annotations(raw_s)
        epochs_s = mne.Epochs(
            raw_s,
            events_s,
            event_id={'T1': 2, 'T2': 3},
            tmin=0,
            tmax=3,
            baseline=None,
            preload=True
        )

        data_s = epochs_s.get_data()
        ptp_s = (data_s.max(axis=2) - data_s.min(axis=2)).max(axis=1)
        all_ptp_subject.extend(ptp_s * 1e6)

    all_ptp_subject = np.array(all_ptp_subject)
    subject_median = np.median(all_ptp_subject)
    subject_results[subject] = {
        'median': subject_median,
        'ptp': all_ptp_subject
    }

    print(f"{subject:<10} {subject_median:<12.1f}", end="")
    for m in multipliers:
        threshold = subject_median * m
        rejected = (all_ptp_subject > threshold).sum()
        pct_kept = (1 - rejected / len(all_ptp_subject)) * 100
        print(f"{threshold:<12.1f} {rejected:<10} {pct_kept:<10.1f}", end="")
    print()


### hybrid threshold: absolute ceiling + relative multiplier

In [ ]:
# absolute ceiling catches extreme outliers in noisy subjects
# relative multiplier handles low amplitude subjects

absolute_ceiling = 300  # uV
relative_multiplier = 3

print(f"Hybrid threshold: absolute {absolute_ceiling} uV OR {relative_multiplier}x subject median")
print(f"\n{'Subject':<10} {'Median uV':<12} {'Rel thresh':<12} {'Rejected':<10} {'Pct kept':<10}")
print("-" * 56)

for subject, results in subject_results.items():
    median = results['median']
    ptp = results['ptp']
    rel_threshold = median * relative_multiplier

    rejected = ((ptp > absolute_ceiling) | (ptp > rel_threshold)).sum()
    pct_kept = (1 - rejected / len(ptp)) * 100

    print(
        f"{subject:<10} {median:<12.1f} {rel_threshold:<12.1f} "
        f"{rejected:<10} {pct_kept:<10.1f}"
    )

### hybrid threshold exploration

In [ ]:
# combines absolute ceiling with relative multiplier
# absolute ceiling catches extreme outliers in noisy subjects
# relative multiplier handles low amplitude subjects
# testing a few combinations to find the most balanced approach

absolute_ceilings = [200, 250, 300]
relative_multipliers = [2, 3, 4]

for absolute_ceiling in absolute_ceilings:
    for relative_multiplier in relative_multipliers:
        print(f"\nHybrid: absolute {absolute_ceiling} uV OR {relative_multiplier}x subject median")
        print(f"{'Subject':<10} {'Median uV':<12} {'Rel thresh':<12} {'Rejected':<10} {'Pct kept':<10}")
        print("-" * 56)

        total_rejected = 0
        total_trials = 0

        for subject, results in subject_results.items():
            median = results['median']
            ptp = results['ptp']
            rel_threshold = median * relative_multiplier

            rejected = ((ptp > absolute_ceiling) |
                       (ptp > rel_threshold)).sum()
            pct_kept = (1 - rejected / len(ptp)) * 100
            total_rejected += rejected
            total_trials += len(ptp)

            print(
                f"{subject:<10} {median:<12.1f} {rel_threshold:<12.1f} "
                f"{rejected:<10} {pct_kept:<10.1f}"
            )

        overall_pct = (1 - total_rejected / total_trials) * 100
        print(f"{'Overall':<10} {'':<12} {'':<12} {total_rejected:<10} {overall_pct:<10.1f}")

### automated ICA exploration

In [ ]:
subjects_to_check = ['S001', 'S010', 'S020', 'S030', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']

ica_results = {}  # must be outside the loop

for subject in subjects_to_check:
    print(f"\nProcessing {subject}...")

    subject_raw_broad = []
    subject_raw_narrow = []

    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"

        # broad filter version for ICA fitting
        raw_broad = mne.io.read_raw_edf(edf_path, preload=True)
        raw_broad.notch_filter(freqs=60)
        raw_broad.filter(l_freq=1.0, h_freq=79.0, method='fir', fir_design='firwin')
        raw_broad.set_eeg_reference('average', projection=False)
        subject_raw_broad.append(raw_broad)

        # narrow filter version for analysis
        raw_narrow = mne.io.read_raw_edf(edf_path, preload=True)
        raw_narrow.notch_filter(freqs=60)
        raw_narrow.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_narrow.set_eeg_reference('average', projection=False)
        subject_raw_narrow.append(raw_narrow)

    # concatenate both versions
    raw_broad_concat = mne.concatenate_raws(subject_raw_broad)
    raw_narrow_concat = mne.concatenate_raws(subject_raw_narrow)

    # rename channels for montage compatibility
    montage = mne.channels.make_standard_montage('standard_1020')
    montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

    def fix_ch(name):
        upper = name.upper()
        return montage_lookup[upper] if upper in montage_lookup else name

    # rename broad concat for ICA fitting
    raw_broad_renamed = raw_broad_concat.copy()
    rename_map = {ch: ch.rstrip('.') for ch in raw_broad_renamed.ch_names}
    raw_broad_renamed.rename_channels(rename_map)
    cap_fix = {ch: fix_ch(ch) for ch in raw_broad_renamed.ch_names}
    raw_broad_renamed.rename_channels(cap_fix)
    raw_broad_renamed.set_montage(montage, on_missing='ignore')

    # rename narrow concat for analysis
    raw_narrow_renamed = raw_narrow_concat.copy()
    rename_map_narrow = {ch: ch.rstrip('.') for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(rename_map_narrow)
    cap_fix_narrow = {ch: fix_ch(ch) for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(cap_fix_narrow)
    raw_narrow_renamed.set_montage(montage, on_missing='ignore')

    # fit ICA on broad filtered signal using extended infomax
    ica = ICA(
        n_components=20,
        random_state=25,
        method='infomax',
        fit_params=dict(extended=True)
    )
    ica.fit(raw_broad_renamed)

    # classify components using icalabel
    ic_labels = label_components(raw_broad_renamed, ica, method='iclabel')
    labels = ic_labels['labels']
    probabilities = ic_labels['y_pred_proba']

    # identify components to exclude
    exclude_idx = []
    for idx, (label, prob) in enumerate(zip(labels, probabilities)):
        if label != 'brain' and prob.max() > 0.7:
            exclude_idx.append(idx)

    print(f"  Components fitted: 20")
    print(f"  Components to exclude: {len(exclude_idx)}")
    print(f"  Labels: {labels}")

    # apply ICA to narrow filtered signal for analysis
    ica.exclude = exclude_idx
    ica.apply(raw_narrow_renamed)

    # epoch the cleaned narrow signal
    events_clean, _ = mne.events_from_annotations(raw_narrow_renamed)
    epochs_clean = mne.Epochs(
        raw_narrow_renamed,
        events_clean,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    data_clean = epochs_clean.get_data()
    ptp_clean = (data_clean.max(axis=2) - data_clean.min(axis=2)).max(axis=1)

    ica_results[subject] = {
        'n_excluded': len(exclude_idx),
        'labels': labels,
        'ptp_after_ica': ptp_clean * 1e6,
        'n_trials': len(ptp_clean)
    }

# summary outside loop
print("\nICA summary:")
print(f"{'Subject':<10} {'Excluded':<10} {'Trials':<10} {'Max uV after ICA':<20} {'Above 250uV':<15}")
print("-" * 67)
for subject, results in ica_results.items():
    ptp = results['ptp_after_ica']
    above_250 = (ptp > 250).sum()
    print(
        f"{subject:<10} {results['n_excluded']:<10} "
        f"{results['n_trials']:<10} {ptp.max():<20.1f} "
        f"{above_250:<15}"
    )

### ICA MNE without icalabel - final post v2 checks

In [ ]:
subjects_to_check = ['S001', 'S010', 'S020', 'S030', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']

ica_opta_results = {}

for subject in subjects_to_check:
    print(f"\nProcessing {subject}...")

    subject_raw_list = []

    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=1.0, h_freq=79.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)
        subject_raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(subject_raw_list)

    # rename channels for montage
    montage = mne.channels.make_standard_montage('standard_1020')
    montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

    def fix_ch(name):
        upper = name.upper()
        return montage_lookup[upper] if upper in montage_lookup else name

    raw_renamed = raw_concat.copy()
    rename_map = {ch: ch.rstrip('.') for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(rename_map)
    cap_fix = {ch: fix_ch(ch) for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(cap_fix)
    raw_renamed.set_montage(montage, on_missing='ignore')

    # fit ICA using extended infomax
    ica = ICA(
        n_components=20,
        random_state=25,
        method='infomax',
        fit_params=dict(extended=True)
    )
    ica.fit(raw_renamed)

    # option A: detect artifacts using frontal channels as EOG proxy
    # Fp1 and Fp2 are frontal channels that pick up eye movements
    # no dedicated EOG channel so we use these as proxy
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_renamed,
        ch_name=['Fp1', 'Fp2'],
        threshold=3.0
    )

    # # detect muscle artifacts using z-score threshold on component variance
    # muscle_indices, muscle_scores = ica.find_bads_muscle(
    #     raw_renamed,
    #     threshold=1.0
    # )

    # combine all artifact components
    exclude_idx = eog_indices
    print(f"  EOG components found: {len(eog_indices)} {eog_indices}")
    print(f"  Total to exclude: {len(exclude_idx)}")

    # apply to narrow filtered signal
    raw_narrow_list = []
    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_n = mne.io.read_raw_edf(edf_path, preload=True)
        raw_n.notch_filter(freqs=60)
        raw_n.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_n.set_eeg_reference('average', projection=False)
        raw_narrow_list.append(raw_n)

    raw_narrow_concat = mne.concatenate_raws(raw_narrow_list)
    raw_narrow_renamed = raw_narrow_concat.copy()
    rename_map_n = {ch: ch.rstrip('.') for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(rename_map_n)
    cap_fix_n = {ch: fix_ch(ch) for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(cap_fix_n)
    raw_narrow_renamed.set_montage(montage, on_missing='ignore')

    ica.exclude = exclude_idx
    ica.apply(raw_narrow_renamed)

    # epoch cleaned signal
    events_clean, _ = mne.events_from_annotations(raw_narrow_renamed)
    epochs_clean = mne.Epochs(
        raw_narrow_renamed,
        events_clean,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    data_clean = epochs_clean.get_data()
    ptp_clean = (data_clean.max(axis=2) - data_clean.min(axis=2)).max(axis=1)

    ica_opta_results[subject] = {
        'n_eog': len(eog_indices),
        # 'n_muscle': len(muscle_indices),
        'n_excluded': len(exclude_idx),
        'ptp_after_ica': ptp_clean * 1e6,
        'n_trials': len(ptp_clean)
    }

print("\nOption A ICA summary:")
print(f"{'Subject':<10} {'EOG':<8} {'Muscle':<10} {'Excluded':<10} {'Trials':<10} {'Max uV':<12} {'Above 250uV':<12}")
print("-" * 74)
for subject, results in ica_opta_results.items():
    ptp = results['ptp_after_ica']
    above_250 = (ptp > 250).sum()
    print(
        f"{subject:<10} {results['n_eog']:<8} "
        # f"{results['n_muscle']:<10} {results['n_excluded']:<10} "
        f"{results['n_trials']:<10} {ptp.max():<12.1f} "
        f"{above_250:<12}"
    )

In [ ]:
# verify ica_opta_results contains EOG only run
for subject, results in ica_opta_results.items():
    print(f"{subject}: excluded={results['n_excluded']}")

### V2 with threshold 1.0

In [ ]:
subjects_to_check = ['S001', 'S010', 'S020', 'S030', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']

ica_opta_results_v2 = {}

for subject in subjects_to_check:
    print(f"\nProcessing {subject}...")

    subject_raw_list = []

    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=1.0, h_freq=79.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)
        subject_raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(subject_raw_list)

    # rename channels for montage
    montage = mne.channels.make_standard_montage('standard_1020')
    montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

    def fix_ch(name):
        upper = name.upper()
        return montage_lookup[upper] if upper in montage_lookup else name

    raw_renamed = raw_concat.copy()
    rename_map = {ch: ch.rstrip('.') for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(rename_map)
    cap_fix = {ch: fix_ch(ch) for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(cap_fix)
    raw_renamed.set_montage(montage, on_missing='ignore')

    # fit ICA using extended infomax
    ica = ICA(
        n_components=20,
        random_state=25,
        method='infomax',
        fit_params=dict(extended=True)
    )
    ica.fit(raw_renamed)

    # option A: detect artifacts using frontal channels as EOG proxy
    # Fp1 and Fp2 are frontal channels that pick up eye movements
    # no dedicated EOG channel so we use these as proxy
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_renamed,
        ch_name=['Fp1', 'Fp2'],
        threshold=3.0
    )

    # detect muscle artifacts using z-score threshold on component variance
    muscle_indices, muscle_scores = ica.find_bads_muscle(
        raw_renamed,
        threshold=1.0
    )

    # combine all artifact components
    exclude_idx = list(set(eog_indices + muscle_indices))

    print(f"  EOG components found: {len(eog_indices)} {eog_indices}")
    print(f"  Muscle components found: {len(muscle_indices)} {muscle_indices}")
    print(f"  Total to exclude: {len(exclude_idx)}")

    # apply to narrow filtered signal
    raw_narrow_list = []
    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_n = mne.io.read_raw_edf(edf_path, preload=True)
        raw_n.notch_filter(freqs=60)
        raw_n.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_n.set_eeg_reference('average', projection=False)
        raw_narrow_list.append(raw_n)

    raw_narrow_concat = mne.concatenate_raws(raw_narrow_list)
    raw_narrow_renamed = raw_narrow_concat.copy()
    rename_map_n = {ch: ch.rstrip('.') for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(rename_map_n)
    cap_fix_n = {ch: fix_ch(ch) for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(cap_fix_n)
    raw_narrow_renamed.set_montage(montage, on_missing='ignore')

    ica.exclude = exclude_idx
    ica.apply(raw_narrow_renamed)

    # epoch cleaned signal
    events_clean, _ = mne.events_from_annotations(raw_narrow_renamed)
    epochs_clean = mne.Epochs(
        raw_narrow_renamed,
        events_clean,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    data_clean = epochs_clean.get_data()
    ptp_clean = (data_clean.max(axis=2) - data_clean.min(axis=2)).max(axis=1)

    ica_opta_results_v2[subject] = {
        'n_eog': len(eog_indices),
        'n_muscle': len(muscle_indices),
        'n_excluded': len(exclude_idx),
        'ptp_after_ica': ptp_clean * 1e6,
        'n_trials': len(ptp_clean)
    }

print("\nOption A ICA summary v2 (muscle threshold 1.0):")
print(f"{'Subject':<10} {'EOG':<8} {'Muscle':<10} {'Excluded':<10} {'Trials':<10} {'Max uV':<12} {'Above 250uV':<12}")
print("-" * 74)
for subject, results in ica_opta_results_v2.items():
    ptp = results['ptp_after_ica']
    above_250 = (ptp > 250).sum()
    print(
        f"{subject:<10} {results['n_eog']:<8} "
        f"{results['n_muscle']:<10} {results['n_excluded']:<10} "
        f"{results['n_trials']:<10} {ptp.max():<12.1f} "
        f"{above_250:<12}"
    )

#### Threshold 0.6

In [ ]:
subjects_to_check = ['S001', 'S010', 'S020', 'S030', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']

ica_opta_results_v2 = {}

for subject in subjects_to_check:
    print(f"\nProcessing {subject}...")

    subject_raw_list = []

    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=1.0, h_freq=79.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)
        subject_raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(subject_raw_list)

    # rename channels for montage
    montage = mne.channels.make_standard_montage('standard_1020')
    montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

    def fix_ch(name):
        upper = name.upper()
        return montage_lookup[upper] if upper in montage_lookup else name

    raw_renamed = raw_concat.copy()
    rename_map = {ch: ch.rstrip('.') for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(rename_map)
    cap_fix = {ch: fix_ch(ch) for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(cap_fix)
    raw_renamed.set_montage(montage, on_missing='ignore')

    # fit ICA using extended infomax
    ica = ICA(
        n_components=20,
        random_state=25,
        method='infomax',
        fit_params=dict(extended=True)
    )
    ica.fit(raw_renamed)

    # option A: detect artifacts using frontal channels as EOG proxy
    # Fp1 and Fp2 are frontal channels that pick up eye movements
    # no dedicated EOG channel so we use these as proxy
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_renamed,
        ch_name=['Fp1', 'Fp2'],
        threshold=3.0
    )

    # detect muscle artifacts using z-score threshold on component variance
    muscle_indices, muscle_scores = ica.find_bads_muscle(
        raw_renamed,
        threshold=0.6
    )

    # combine all artifact components
    exclude_idx = list(set(eog_indices + muscle_indices))

    print(f"  EOG components found: {len(eog_indices)} {eog_indices}")
    print(f"  Muscle components found: {len(muscle_indices)} {muscle_indices}")
    print(f"  Total to exclude: {len(exclude_idx)}")

    # apply to narrow filtered signal
    raw_narrow_list = []
    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_n = mne.io.read_raw_edf(edf_path, preload=True)
        raw_n.notch_filter(freqs=60)
        raw_n.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_n.set_eeg_reference('average', projection=False)
        raw_narrow_list.append(raw_n)

    raw_narrow_concat = mne.concatenate_raws(raw_narrow_list)
    raw_narrow_renamed = raw_narrow_concat.copy()
    rename_map_n = {ch: ch.rstrip('.') for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(rename_map_n)
    cap_fix_n = {ch: fix_ch(ch) for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(cap_fix_n)
    raw_narrow_renamed.set_montage(montage, on_missing='ignore')

    ica.exclude = exclude_idx
    ica.apply(raw_narrow_renamed)

    # epoch cleaned signal
    events_clean, _ = mne.events_from_annotations(raw_narrow_renamed)
    epochs_clean = mne.Epochs(
        raw_narrow_renamed,
        events_clean,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    data_clean = epochs_clean.get_data()
    ptp_clean = (data_clean.max(axis=2) - data_clean.min(axis=2)).max(axis=1)

    ica_opta_results_v2[subject] = {
        'n_eog': len(eog_indices),
        'n_muscle': len(muscle_indices),
        'n_excluded': len(exclude_idx),
        'ptp_after_ica': ptp_clean * 1e6,
        'n_trials': len(ptp_clean)
    }

print("\nOption A ICA summary v2 (muscle threshold 1.0):")
print(f"{'Subject':<10} {'EOG':<8} {'Muscle':<10} {'Excluded':<10} {'Trials':<10} {'Max uV':<12} {'Above 250uV':<12}")
print("-" * 74)
for subject, results in ica_opta_results_v2.items():
    ptp = results['ptp_after_ica']
    above_250 = (ptp > 250).sum()
    print(
        f"{subject:<10} {results['n_eog']:<8} "
        f"{results['n_muscle']:<10} {results['n_excluded']:<10} "
        f"{results['n_trials']:<10} {ptp.max():<12.1f} "
        f"{above_250:<12}"
    )

#### threshold 0.7

In [ ]:
subjects_to_check = ['S001', 'S010', 'S020', 'S030', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']

ica_opta_results_v2 = {}

for subject in subjects_to_check:
    print(f"\nProcessing {subject}...")

    subject_raw_list = []

    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=1.0, h_freq=79.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)
        subject_raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(subject_raw_list)

    # rename channels for montage
    montage = mne.channels.make_standard_montage('standard_1020')
    montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

    def fix_ch(name):
        upper = name.upper()
        return montage_lookup[upper] if upper in montage_lookup else name

    raw_renamed = raw_concat.copy()
    rename_map = {ch: ch.rstrip('.') for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(rename_map)
    cap_fix = {ch: fix_ch(ch) for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(cap_fix)
    raw_renamed.set_montage(montage, on_missing='ignore')

    # fit ICA using extended infomax
    ica = ICA(
        n_components=20,
        random_state=25,
        method='infomax',
        fit_params=dict(extended=True)
    )
    ica.fit(raw_renamed)

    # option A: detect artifacts using frontal channels as EOG proxy
    # Fp1 and Fp2 are frontal channels that pick up eye movements
    # no dedicated EOG channel so we use these as proxy
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_renamed,
        ch_name=['Fp1', 'Fp2'],
        threshold=3.0
    )

    # detect muscle artifacts using z-score threshold on component variance
    muscle_indices, muscle_scores = ica.find_bads_muscle(
        raw_renamed,
        threshold=0.7
    )

    # combine all artifact components
    exclude_idx = list(set(eog_indices + muscle_indices))

    print(f"  EOG components found: {len(eog_indices)} {eog_indices}")
    print(f"  Muscle components found: {len(muscle_indices)} {muscle_indices}")
    print(f"  Total to exclude: {len(exclude_idx)}")

    # apply to narrow filtered signal
    raw_narrow_list = []
    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_n = mne.io.read_raw_edf(edf_path, preload=True)
        raw_n.notch_filter(freqs=60)
        raw_n.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_n.set_eeg_reference('average', projection=False)
        raw_narrow_list.append(raw_n)

    raw_narrow_concat = mne.concatenate_raws(raw_narrow_list)
    raw_narrow_renamed = raw_narrow_concat.copy()
    rename_map_n = {ch: ch.rstrip('.') for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(rename_map_n)
    cap_fix_n = {ch: fix_ch(ch) for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(cap_fix_n)
    raw_narrow_renamed.set_montage(montage, on_missing='ignore')

    ica.exclude = exclude_idx
    ica.apply(raw_narrow_renamed)

    # epoch cleaned signal
    events_clean, _ = mne.events_from_annotations(raw_narrow_renamed)
    epochs_clean = mne.Epochs(
        raw_narrow_renamed,
        events_clean,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    data_clean = epochs_clean.get_data()
    ptp_clean = (data_clean.max(axis=2) - data_clean.min(axis=2)).max(axis=1)

    ica_opta_results_v2[subject] = {
        'n_eog': len(eog_indices),
        'n_muscle': len(muscle_indices),
        'n_excluded': len(exclude_idx),
        'ptp_after_ica': ptp_clean * 1e6,
        'n_trials': len(ptp_clean)
    }

print("\nOption A ICA summary v2 (muscle threshold 1.0):")
print(f"{'Subject':<10} {'EOG':<8} {'Muscle':<10} {'Excluded':<10} {'Trials':<10} {'Max uV':<12} {'Above 250uV':<12}")
print("-" * 74)
for subject, results in ica_opta_results_v2.items():
    ptp = results['ptp_after_ica']
    above_250 = (ptp > 250).sum()
    print(
        f"{subject:<10} {results['n_eog']:<8} "
        f"{results['n_muscle']:<10} {results['n_excluded']:<10} "
        f"{results['n_trials']:<10} {ptp.max():<12.1f} "
        f"{above_250:<12}"
    )

#### Threshold 0.8

In [ ]:
subjects_to_check = ['S001', 'S010', 'S020', 'S030', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']

ica_opta_results_v2 = {}

for subject in subjects_to_check:
    print(f"\nProcessing {subject}...")

    subject_raw_list = []

    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=1.0, h_freq=79.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)
        subject_raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(subject_raw_list)

    # rename channels for montage
    montage = mne.channels.make_standard_montage('standard_1020')
    montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

    def fix_ch(name):
        upper = name.upper()
        return montage_lookup[upper] if upper in montage_lookup else name

    raw_renamed = raw_concat.copy()
    rename_map = {ch: ch.rstrip('.') for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(rename_map)
    cap_fix = {ch: fix_ch(ch) for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(cap_fix)
    raw_renamed.set_montage(montage, on_missing='ignore')

    # fit ICA using extended infomax
    ica = ICA(
        n_components=20,
        random_state=25,
        method='infomax',
        fit_params=dict(extended=True)
    )
    ica.fit(raw_renamed)

    # option A: detect artifacts using frontal channels as EOG proxy
    # Fp1 and Fp2 are frontal channels that pick up eye movements
    # no dedicated EOG channel so we use these as proxy
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_renamed,
        ch_name=['Fp1', 'Fp2'],
        threshold=3.0
    )

    # detect muscle artifacts using z-score threshold on component variance
    muscle_indices, muscle_scores = ica.find_bads_muscle(
        raw_renamed,
        threshold=0.8
    )

    # combine all artifact components
    exclude_idx = list(set(eog_indices + muscle_indices))

    print(f"  EOG components found: {len(eog_indices)} {eog_indices}")
    print(f"  Muscle components found: {len(muscle_indices)} {muscle_indices}")
    print(f"  Total to exclude: {len(exclude_idx)}")

    # apply to narrow filtered signal
    raw_narrow_list = []
    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_n = mne.io.read_raw_edf(edf_path, preload=True)
        raw_n.notch_filter(freqs=60)
        raw_n.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_n.set_eeg_reference('average', projection=False)
        raw_narrow_list.append(raw_n)

    raw_narrow_concat = mne.concatenate_raws(raw_narrow_list)
    raw_narrow_renamed = raw_narrow_concat.copy()
    rename_map_n = {ch: ch.rstrip('.') for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(rename_map_n)
    cap_fix_n = {ch: fix_ch(ch) for ch in raw_narrow_renamed.ch_names}
    raw_narrow_renamed.rename_channels(cap_fix_n)
    raw_narrow_renamed.set_montage(montage, on_missing='ignore')

    ica.exclude = exclude_idx
    ica.apply(raw_narrow_renamed)

    # epoch cleaned signal
    events_clean, _ = mne.events_from_annotations(raw_narrow_renamed)
    epochs_clean = mne.Epochs(
        raw_narrow_renamed,
        events_clean,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    data_clean = epochs_clean.get_data()
    ptp_clean = (data_clean.max(axis=2) - data_clean.min(axis=2)).max(axis=1)

    ica_opta_results_v2[subject] = {
        'n_eog': len(eog_indices),
        'n_muscle': len(muscle_indices),
        'n_excluded': len(exclude_idx),
        'ptp_after_ica': ptp_clean * 1e6,
        'n_trials': len(ptp_clean)
    }

print("\nOption A ICA summary v2 (muscle threshold 1.0):")
print(f"{'Subject':<10} {'EOG':<8} {'Muscle':<10} {'Excluded':<10} {'Trials':<10} {'Max uV':<12} {'Above 250uV':<12}")
print("-" * 74)
for subject, results in ica_opta_results_v2.items():
    ptp = results['ptp_after_ica']
    above_250 = (ptp > 250).sum()
    print(
        f"{subject:<10} {results['n_eog']:<8} "
        f"{results['n_muscle']:<10} {results['n_excluded']:<10} "
        f"{results['n_trials']:<10} {ptp.max():<12.1f} "
        f"{above_250:<12}"
    )

### Comparison of the ouput of ICA's with Hybrid

In [ ]:
# full comparison: hybrid threshold vs ICA icalabel vs ICA MNE version

print("Full comparison across artifact rejection approaches")
print(f"\n{'Subject':<10} {'Hybrid 250/3x':<16} {'ICA icalabel':<16} {'ICA option A':<16}")
print(f"{'':10} {'above 250uV':<16} {'above 250uV':<16} {'above 250uV':<16}")
print("-" * 60)

hybrid_above_250 = {
    'S001': 2, 'S010': 1, 'S020': 0,
    'S030': 1, 'S050': 0, 'S070': 1, 'S090': 3
}

for subject in subjects_to_check:
    ica_c_above = (ica_results[subject]['ptp_after_ica'] > 250).sum()
    ica_a_above = (ica_opta_results[subject]['ptp_after_ica'] > 250).sum()
    hybrid = hybrid_above_250[subject]
    print(
        f"{subject:<10} {hybrid:<16} {ica_c_above:<16} {ica_a_above:<16}"
    )

print(f"\n{'Subject':<10} {'Hybrid excl':<16} {'ICLabel excl':<16} {'OptA excl':<16}")
print("-" * 60)
for subject in subjects_to_check:
    print(
        f"{subject:<10} {'n/a':<16} "
        f"{ica_results[subject]['n_excluded']:<16} "
        f"{ica_opta_results[subject]['n_excluded']:<16}"
    )

### final comparison: hybrid threshold vs ICA icalabel vs ICA EOG only

In [ ]:
# two stage approach: ICA EOG removal first, then hybrid threshold as safety net

print("Final artifact rejection comparison")
print(f"\n{'Subject':<10} {'Hybrid 250/3x':<16} {'ICA icalabel':<16} {'ICA EOG only':<16}")
print(f"{'':10} {'above 250uV':<16} {'above 250uV':<16} {'above 250uV':<16}")
print("-" * 60)

hybrid_above_250 = {
    'S001': 2, 'S010': 1, 'S020': 0,
    'S030': 1, 'S050': 0, 'S070': 1, 'S090': 3
}

for subject in subjects_to_check:
    ica_c_above = (ica_results[subject]['ptp_after_ica'] > 250).sum()
    ica_eog_above = (ica_opta_results[subject]['ptp_after_ica'] > 250).sum()
    hybrid = hybrid_above_250[subject]
    print(
        f"{subject:<10} {hybrid:<16} {ica_c_above:<16} {ica_eog_above:<16}"
    )

print(f"\n{'Subject':<10} {'ICA icalabel':<16} {'ICA EOG only':<16}")
print(f"{'':10} {'comps excluded':<16} {'comps excluded':<16}")
print("-" * 44)
for subject in subjects_to_check:
    print(
        f"{subject:<10} {ica_results[subject]['n_excluded']:<16} "
        f"{ica_opta_results[subject]['n_excluded']:<16}"
    )

### Channel Selection

In [ ]:
# checking whether neighbouring channels add useful information
# beyond C3, Cz, C4 for motor imagery classification

# candidate channels: motor cortex core + surrounding
core_channels = ['C3', 'Cz', 'C4']
neighbouring_channels = ['FC3', 'FCz', 'FC4', 'CP3', 'CPz', 'CP4']
all_candidate_channels = core_channels + neighbouring_channels

subjects_to_check = ['S001', 'S010', 'S020', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']

channel_variance = {ch: {'T1': [], 'T2': []} for ch in all_candidate_channels}

for subject in subjects_to_check:
    raw_list = []
    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)
        raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(raw_list)

    # rename channels
    montage = mne.channels.make_standard_montage('standard_1020')
    montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

    def fix_ch(name):
        upper = name.upper()
        return montage_lookup[upper] if upper in montage_lookup else name

    raw_renamed = raw_concat.copy()
    rename_map = {ch: ch.rstrip('.') for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(rename_map)
    cap_fix = {ch: fix_ch(ch) for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(cap_fix)
    raw_renamed.set_montage(montage, on_missing='ignore')

    # epoch
    events_s, _ = mne.events_from_annotations(raw_renamed)
    epochs_s = mne.Epochs(
        raw_renamed,
        events_s,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    # variance per channel per condition
    for ch in all_candidate_channels:
        if ch in epochs_s.ch_names:
            ch_idx = epochs_s.ch_names.index(ch)
            t1_var = epochs_s['T1'].get_data()[:, ch_idx, :].var(axis=1).mean()
            t2_var = epochs_s['T2'].get_data()[:, ch_idx, :].var(axis=1).mean()
            channel_variance[ch]['T1'].append(t1_var)
            channel_variance[ch]['T2'].append(t2_var)

# print mean variance per channel across subjects
print(f"{'Channel':<10} {'T1 mean var':>14} {'T2 mean var':>14} {'T1/T2 ratio':>14} {'Group':<12}")
print("-" * 66)

for ch in all_candidate_channels:
    t1_mean = np.mean(channel_variance[ch]['T1'])
    t2_mean = np.mean(channel_variance[ch]['T2'])
    ratio = t1_mean / t2_mean
    group = 'core' if ch in core_channels else 'neighbouring'
    print(
        f"{ch:<10} {t1_mean:>14.2f} {t2_mean:>14.2f} "
        f"{ratio:>14.3f} {group:<12}"
    )

In [ ]:
# check lateralisation consistency across subjects for all candidate channels

all_candidate_channels = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'FC4', 'CP3', 'CPz', 'CP4']

print(f"{'Subject':<10}", end="")
for ch in all_candidate_channels:
    print(f"{ch:>10}", end="")
print()
print("-" * 100)

for i, subject in enumerate(subjects_to_check):
    print(f"{subject:<10}", end="")
    for ch in all_candidate_channels:
        t1 = channel_variance[ch]['T1'][i]
        t2 = channel_variance[ch]['T2'][i]
        ratio = t1 / t2
        print(f"{ratio:>10.3f}", end="")
    print()

# also print mean ratio per channel at the bottom for reference
print("-" * 100)
print(f"{'Mean':<10}", end="")
for ch in all_candidate_channels:
    ratios = [channel_variance[ch]['T1'][i] / channel_variance[ch]['T2'][i]
              for i in range(len(subjects_to_check))]
    print(f"{np.mean(ratios):>10.3f}", end="")
print()

print(f"{'Std':<10}", end="")
for ch in all_candidate_channels:
    ratios = [channel_variance[ch]['T1'][i] / channel_variance[ch]['T2'][i]
              for i in range(len(subjects_to_check))]
    print(f"{np.std(ratios):>10.3f}", end="")
print()

### Normalisation exploration

In [ ]:
# comparing z-score, min-max, robust scaling, global z-score
# checking cross-subject amplitude consistency and C3 vs C4 separation

subjects_to_check = ['S001', 'S020', 'S050', 'S070', 'S090']
runs = ['R04', 'R08', 'R12']
selected_channels = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4']

# collect epochs for each subject after ICA and before normalisation
subject_epochs = {}

for subject in subjects_to_check:
    raw_list = []
    for run in runs:
        edf_path = data_path / subject / f"{subject}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True)
        raw_s.notch_filter(freqs=60)
        raw_s.filter(l_freq=8.0, h_freq=30.0, method='fir', fir_design='firwin')
        raw_s.set_eeg_reference('average', projection=False)
        raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(raw_list)

    # rename channels
    montage = mne.channels.make_standard_montage('standard_1020')
    montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

    def fix_ch(name):
        upper = name.upper()
        return montage_lookup[upper] if upper in montage_lookup else name

    raw_renamed = raw_concat.copy()
    rename_map = {ch: ch.rstrip('.') for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(rename_map)
    cap_fix = {ch: fix_ch(ch) for ch in raw_renamed.ch_names}
    raw_renamed.rename_channels(cap_fix)
    raw_renamed.set_montage(montage, on_missing='ignore')

    # apply EOG only ICA
    ica = ICA(n_components=20, random_state=25, method='infomax',
              fit_params=dict(extended=True))

    # fit on broad filtered version
    raw_broad = raw_concat.copy()
    raw_broad.notch_filter(freqs=60)
    raw_broad.filter(l_freq=1.0, h_freq=79.0, method='fir', fir_design='firwin')
    raw_broad.set_eeg_reference('average', projection=False)

    raw_broad_renamed = raw_broad.copy()
    rename_map_b = {ch: ch.rstrip('.') for ch in raw_broad_renamed.ch_names}
    raw_broad_renamed.rename_channels(rename_map_b)
    cap_fix_b = {ch: fix_ch(ch) for ch in raw_broad_renamed.ch_names}
    raw_broad_renamed.rename_channels(cap_fix_b)
    raw_broad_renamed.set_montage(montage, on_missing='ignore')

    ica.fit(raw_broad_renamed)
    eog_indices, _ = ica.find_bads_eog(raw_broad_renamed,
                                        ch_name=['Fp1', 'Fp2'],
                                        threshold=3.0)
    ica.exclude = eog_indices
    ica.apply(raw_renamed)

    # epoch
    events_s, _ = mne.events_from_annotations(raw_renamed)
    epochs_s = mne.Epochs(
        raw_renamed,
        events_s,
        event_id={'T1': 2, 'T2': 3},
        tmin=0,
        tmax=3,
        baseline=None,
        preload=True
    )

    subject_epochs[subject] = epochs_s

print("Epochs collected for all subjects")
for subject, epochs in subject_epochs.items():
    print(f"  {subject}: {len(epochs)} trials")

In [ ]:
print(list(subject_epochs.keys()))
print(subject_epochs['S001'].ch_names[:10])

In [ ]:
# normalisation exploration
# comparing z-score per channel, min-max per channel,
# robust scaling per channel, global z-score across channels

from sklearn.preprocessing import RobustScaler

def apply_zscore(data):
    # data shape: trials x channels x timepoints
    # z-score per channel across all timepoints and trials
    mean = data.mean(axis=(0, 2), keepdims=True)
    std = data.std(axis=(0, 2), keepdims=True)
    return (data - mean) / (std + 1e-8)

def apply_minmax(data):
    min_val = data.min(axis=(0, 2), keepdims=True)
    max_val = data.max(axis=(0, 2), keepdims=True)
    return (data - min_val) / (max_val - min_val + 1e-8)

def apply_robust(data):
    # robust scaling per channel using median and IQR
    trials, channels, timepoints = data.shape
    result = np.zeros_like(data)
    for ch in range(channels):
        ch_data = data[:, ch, :].flatten()
        median = np.median(ch_data)
        q75, q25 = np.percentile(ch_data, [75, 25])
        iqr = q75 - q25 + 1e-8
        result[:, ch, :] = (data[:, ch, :] - median) / iqr
    return result

def apply_global_zscore(data):
    # z-score across all channels and timepoints together
    mean = data.mean()
    std = data.std() + 1e-8
    return (data - mean) / std

methods = {
    'z-score per channel': apply_zscore,
    'min-max per channel': apply_minmax,
    'robust per channel': apply_robust,
    'global z-score': apply_global_zscore
}

# for each method, check:
# 1. std of mean amplitude across subjects (lower = more consistent)
# 2. C3 vs C4 variance ratio preservation

print("Cross-subject amplitude consistency after normalisation")
print("Metric: std of mean absolute amplitude across subjects")
print("Lower std = more consistent amplitude across subjects")
print()
print(f"{'Method':<25} {'Std across subjects':>20}")
print("-" * 47)

ratio_results = {}

for method_name, method_fn in methods.items():
    subject_mean_amps = []
    subject_ratios = []

    for subject, epochs in subject_epochs.items():
        # get data for selected channels only
        picks = [epochs.ch_names.index(ch) for ch in selected_channels
                 if ch in epochs.ch_names]
        data = epochs.get_data()[:, picks, :]

        # apply normalisation
        data_norm = method_fn(data)

        # mean absolute amplitude across all trials and timepoints
        mean_amp = np.abs(data_norm).mean()
        subject_mean_amps.append(mean_amp)

        # C3 vs C4 variance ratio after normalisation
        c3_idx = selected_channels.index('C3')
        c4_idx = selected_channels.index('C4')

        t1_indices = [i for i, label in enumerate(epochs.events[:, 2]) if label == 2]
        t2_indices = [i for i, label in enumerate(epochs.events[:, 2]) if label == 3]

        t1_c3_var = data_norm[t1_indices, c3_idx, :].var()
        t1_c4_var = data_norm[t1_indices, c4_idx, :].var()
        t2_c3_var = data_norm[t2_indices, c3_idx, :].var()
        t2_c4_var = data_norm[t2_indices, c4_idx, :].var()

        t1_ratio = t1_c3_var / t1_c4_var
        t2_ratio = t2_c3_var / t2_c4_var
        subject_ratios.append((t1_ratio, t2_ratio))

    std_across_subjects = np.std(subject_mean_amps)
    ratio_results[method_name] = subject_ratios

    print(f"{method_name:<25} {std_across_subjects:>20.6f}")

# lateralisation preservation
print("\nC3 vs C4 variance ratio after normalisation")
print("T1 ratio should be above 1.0, T2 ratio should be below 1.0")
print()
print(f"{'Method':<25} {'Mean T1 ratio':>15} {'Mean T2 ratio':>15} {'Difference':>12}")
print("-" * 69)

for method_name, ratios in ratio_results.items():
    t1_ratios = [r[0] for r in ratios]
    t2_ratios = [r[1] for r in ratios]
    mean_t1 = np.mean(t1_ratios)
    mean_t2 = np.mean(t2_ratios)
    diff = mean_t1 - mean_t2
    print(
        f"{method_name:<25} {mean_t1:>15.3f} "
        f"{mean_t2:>15.3f} {diff:>12.3f}"
    )

### Final decisions mentioned in 01_preprocessing.ipynb

### Multi class

In [ ]:
import mne

# check R06 (both fists/both feet) event structure for one subject
subject_id = 'S001'
edf_path = data_path / subject_id / f"{subject_id}R06.edf"
raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
events, event_id = mne.events_from_annotations(raw, verbose=False)

print(f"Event ID mapping (R06): {event_id}")
print(f"Unique event codes found: {set(events[:, -1])}")
print(f"Event counts: {[(code, (events[:,-1]==code).sum()) for code in set(events[:,-1])]}")

In [ ]:
for run in ['R10', 'R14']:
    edf_path = data_path / subject_id / f"{subject_id}{run}.edf"
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    print(f"{run}: event_id={event_id}, counts={[(code, (events[:,-1]==code).sum()) for code in set(events[:,-1])]}")